# 04 — Data Quality Assessment

> **Retrospective reconstruction notice**
>
> This notebook was prepared retrospectively from available project specifications, phase reports, repository artefacts and conversation records. It is not an original notebook executed during the historical phase.
>
> Evidence labels:
> - **Historically evidenced** — explicitly supported by an available project report, configuration, manifest, source file or recorded execution output.
> - **Reconstructed** — organised or interpreted retrospectively from available evidence.
> - **Not evidenced / not reproducible** — the available evidence does not establish the original execution, exact result or complete decision trail.
>
> This notebook is documentary unless a cell explicitly states otherwise. It does not download data, modify Raw files or claim that illustrative code has been executed.

## 1. Notebook purpose

Consolidate the Phase 3 data quality findings retrospectively. This notebook documents reported validation outcomes, point-in-time classifications and limitations. It does not re-run the original quality tests, modify source data or claim to reproduce the original Phase 3 notebook execution.

## 2. Assessment principles

- Raw files remain immutable.
- Detect and document anomalies; do not silently repair them.
- Separate technical file integrity from semantic and financial data quality.
- Separate data quality acceptance from point-in-time eligibility.
- Separate point-in-time eligibility from licensing eligibility.
- Preserve source provenance and acquisition metadata.
- Do not treat a successful acquisition as proof of suitability for modelling or trading.

## 3. Equity and index market data

The Phase 3 report records 37,409 acquired observations across NG.L, SSE.L, SVT.L and FTSE 100.

| Dataset | Data quality classification | Reported findings |
|---|---|---|
| NG.L | Accept with limitations | Missing and zero-volume observations; OHLC inconsistencies; corporate-action limitations; historical price discrepancy; rights-issue concerns; GBP/GBp denomination handling |
| SSE.L | Accept with limitations | Missing and zero-volume observations; OHLC inconsistencies; corporate-action limitations; denomination checks |
| SVT.L | Accept with limitations | Missing and zero-volume observations; OHLC inconsistencies; corporate-action limitations; denomination checks |
| FTSE 100 | Accept with limitations | CSV/Parquet consistency and file integrity reported; retain provider provenance and applicable market-data limitations |

### Specific equity cautions

- `Adjusted Close` and `Close` are not interchangeable.
- Yahoo Finance corporate-action information was found to be incomplete.
- Historical NG.L rights issues require particular attention.
- Price denomination requires explicit GBP/GBp handling.
- Do not infer that all historical prices are fully adjusted or suitable for accounting solely from provider fields.

**PIT classification note:** The Phase 3 equity classification table assigns `Accept with limitations` as the data-quality classification. The cited report does not assign a formal `PIT_VALIDATED`, `PIT_CONDITIONAL` or `PIT_UNVALIDATED` label to each equity/index dataset in that table. This notebook therefore does not invent equity PIT classifications.

## 4. Interest-rate data

| Dataset | Data quality | PIT classification | Reported issue |
|---|---|---|---|
| Bank Rate | Accept with limitations | `PIT_CONDITIONAL` | Two complete and 15 partial acquisitions; the 15 partial acquisitions were incorrectly marked successful; publication-time limitations |
| SONIA | Accept with limitations | `PIT_CONDITIONAL` | Methodology change effective 23 April 2018; conservative BoE database-series availability policy |
| Gilt 2Y | Accept with limitations | `PIT_UNVALIDATED` | 390 missing observations; historical availability and vintages unverified |
| Gilt 10Y | Accept with limitations | `PIT_UNVALIDATED` | 390 missing observations; historical availability and vintages unverified |

### Additional reported detail

- Bank Rate: 13,104 observations; no invalid dates or duplicate observations; no missing or non-numeric values in the complete files.
- SONIA: 7,496 observations; no invalid dates or duplicate observations; no missing or non-numeric values.
- Gilt curves: eight original workbooks; 12,480 unique dated observations; 12,090 valid observations per maturity; 390 missing observations per maturity, with identical missing dates.

## 5. Macroeconomic data

| Dataset | Reported frequency | Data quality | PIT classification |
|---|---|---|---|
| CPI | Annual, quarterly and monthly | Accept with limitations | `PIT_CONDITIONAL` |
| Core CPI | Annual, quarterly and monthly | Accept with limitations | `PIT_CONDITIONAL` |
| CPIH | Annual, quarterly and monthly | Accept with limitations | `PIT_CONDITIONAL` |
| Real GDP | Annual and quarterly | Accept with limitations | `PIT_UNVALIDATED` |
| GDP Growth | Quarterly | Accept with limitations | `PIT_UNVALIDATED` |

### Main findings

- Source files contain overlapping frequency representations.
- All five series were structurally inspected.
- No automatic transformations or frequency conversions were performed.
- Current ONS files do not provide complete historical publication timestamps for every observation.
- Historical vintages have not been fully reconstructed.
- GDP datasets are not eligible for strict PIT backtesting until the historical vintage limitations are resolved.

## 6. File integrity evidence

The Phase 3 report records:

| Indicator | Result |
|---|---:|
| Manifests checked | 25 |
| Files verified | 29 |
| Missing files | 0 |
| Size mismatches | 0 |
| SHA-256 mismatches | 0 |

Physical file integrity was confirmed for the referenced acquisition files. This does **not** prove that every market observation is economically correct, complete, correctly adjusted or point-in-time safe.

## 7. Calendar and temporal quality

The Phase 3 report records the following operational conventions:

| Parameter | Policy |
|---|---|
| Primary trading calendar | LSE (`XLON`) |
| Operational timezone | `Europe/London` |
| Internal timestamp storage | UTC |
| Trading cut-off | 17:00 `Europe/London` |
| Signal generation | After market close |
| Earliest execution | Next LSE session |

GMT/BST transitions were validated. Historical exceptional LSE closures and calendar-library historical coverage remain documented limitations.

These conventions are temporal controls, not a complete transaction execution model.

## 8. Point-in-time classification policy

| Classification | Research | Model training | Strict PIT backtesting |
|---|---|---|---|
| `PIT_VALIDATED` | Allowed | Allowed | Allowed |
| `PIT_CONDITIONAL` | Allowed | With restrictions | Conservative policy only |
| `PIT_UNVALIDATED` | Allowed with documented limitations | With documented limitations, subject to phase-specific restrictions | Not allowed |

The classifications above reproduce the Phase 3 framework. Dataset-specific restrictions take precedence over a generic interpretation of the table.

PIT eligibility and licensing eligibility must be assessed independently. The existence of a PIT classification does not confer commercial or redistribution rights.

## 9. Open risks carried forward

The Phase 3 report identifies, among others:

| Risk ID | Description | Priority | Status |
|---|---|---|---|
| R-P3-001 | Partial Bank Rate acquisitions | Medium | Open |
| R-P3-009 | Historical dividend discrepancies | Medium | Open |
| R-P3-010 | Incomplete Yahoo Finance corporate actions | High | Open |
| R-P3-011 | NG.L historical price discrepancy | Medium | Open |
| R-P3-021 | Gilt historical vintages | High | Open |
| R-P3-022 | Bank Rate publication timestamps | Medium | Open |
| R-P3-023 | Exceptional historical LSE closures | Medium | Open |
| R-P3-024 | Exchange calendar library changes | Low | Open |
| R-P3-025 | Macroeconomic frequency representation | Medium | Open |

This is a selected list, not the complete project risk register. Other partially resolved findings remain documented in the Phase 3 report and project risk records.

## 10. Subsequent repository inspection — integrity verifier scope

A later inspection of `scripts/verify_acquisition_integrity.py` found that its `ACQUISITIONS` mapping contained SSE.L, SVT.L and `^FTSE`, but not NG.L. A subsequent execution of that script reported successful checks for those three configured assets only.

That execution must not be interpreted as a fresh verification of all four equity/index datasets. It is a limitation of the verifier's configured scope observed after the Phase 3 report. No code change is made by this notebook.

## 11. Evidence classification

- **Historically evidenced:** counts, data-quality classifications, PIT classifications, limitations, calendar policies, file-integrity results and risk identifiers reproduced from the Phase 3 final report.
- **Reconstructed:** consolidation of the report into this notebook structure and explanatory commentary.
- **Not evidenced / not reproducible:** exact cell-by-cell execution of an original Phase 3 notebook, which was not present in the reviewed project artefacts.

Subsequent verifier-scope observations are explicitly distinguished from the Phase 3 report findings.

## 12. Conclusion

The datasets were accepted with limitations, not declared pristine. Downstream exploratory analysis must expose these limitations and must not silently clean, adjust or normalise Raw observations.

Phase 3 permits progression to exploratory data analysis under the dataset-specific restrictions in the final report. It does not automatically approve unrestricted model training, strict PIT backtesting or commercial use.